In [ ]:
import torch

In [ ]:
x = torch.tensor([2.0]) ;x.requires_grad_(True)
print(x)

In [ ]:
y = x ** 2 + 3  
y
print(x.grad)

In [ ]:
y.backward()       # find the derivative we know the derivate of x^2 +3 is 2x = 4 
print(x.grad)


In [ ]:
import torch.nn as nn 

class Neural_Network(nn.Module):
    def __init__(self):
        super(Neural_Network, self).__init__()
        self.layer1 = nn.Linear(64, 32)
        self.layer2 = nn.Linear(32, 10)
        self.relu = nn.ReLU()
    def forward(self,x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x
    
model = Neural_Network()
model

In [ ]:
# steps
# 1. Define the model (input, output size, forward pass)
# 2. Loss and optimizer
# 3. Training loop
#    - forward pass: compute predictions
#    - backward pass: compute gradients
#    - update weights

import torch.optim as optim


optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

x_train = torch.randn(100, 64)  # 100 samples, 64 features            features must be equal to input of first layer
y_train = torch.randint(0, 10, (100,))  # 100                         means 10 classes and 100 samples

optimizer.zero_grad()   # initially pytorch has some weights so we need to zero them out
prediction = model(x_train)  # forward pass
loss = criterion(prediction, y_train)  # compute loss
loss.backward()  # it will find the gradients of all the nodes

optimizer.step()  # update weights

print(f'Loss: {loss.item()}')




## Mnist data prediction using pytorch

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

Transform = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize((0.1307,), (0.3081))
])

train_data = datasets.MNIST(root='./data' , train=True, download=True, transform=Transform)
test_data = datasets.MNIST(root = './data', train = False, download = True, transform=Transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data, batch_size=1000, shuffle=False)

# use CNN Model 

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(64*7*7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout = nn.Dropout(0.25)
    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = x.view(-1, 64*7*7)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x
    
model = CNN()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def train(model, device, train_loader, optimizer, criterion, epoch = 100):
    model.train()
    for epoch in range(1, epoch + 1):
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            if batch_idx % 100 == 0:
                print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} ({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}')





In [ ]:
train(model, device, train_loader  , optimizer, criterion, epoch=100)

In [ ]:
torch.save(model.state_dict(), "mnist_cnn.pth")
